<a href="https://colab.research.google.com/github/yutaota/intro_to_causal_inference/blob/main/Ex8_RDD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 回帰不連続デザイン

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
# RDDのための人工データ生成
np.random.seed(42)
N = 2000  # サンプルサイズ
cutoff = 50  # カットオフ点

# 試験点数 (X): 連続的な共変量
X = np.random.uniform(0, 100, N)

# 処置変数 (A): 試験点数がカットオフ以上なら大学入学 (A=1)、未満なら入学しない (A=0)
A = (X >= cutoff).astype(int)

# アウトカム変数 (Y): 卒業後年収
# YはXに依存し、カットオフ点でジャンプがある
# 真の処置効果を表現するために、カットオフを超えた場合に加算される項を設ける
tau = 150  # 平均処置効果 (大学入学による年収増加)
beta_X = 5  # 試験点数が1点上がると年収が5万円増加する効果
beta_XA = 3  # 試験点数が1点上がると年収が5万円増加する効果
noise = np.random.normal(0, 50, N) # ノイズ

Y = beta_X * X + tau * A + beta_XA * X * A + noise

# データをDataFrameにまとめる
data = pd.DataFrame({'X': X, 'A': A, 'Y': Y})

# データを表示
print(data.head())

# グラフに図示
plt.figure(figsize=(10, 6))
plt.scatter(data['X'], data['Y'], alpha=0.5, label='Data')
plt.axvline(cutoff, color='r', linestyle='--', label='Cutoff')

# 回帰線をプロット (カットオフの左右で別に)
X_below = data[data['X'] < cutoff]['X']
Y_below = data[data['X'] < cutoff]['Y']
X_above = data[data['X'] >= cutoff]['X']
Y_above = data[data['X'] >= cutoff]['Y']

# カットオフ以下の線形回帰
if not X_below.empty:
    model_below = np.polyfit(X_below, Y_below, 1)
    plt.plot(X_below, np.polyval(model_below, X_below), color='blue', label='Below Cutoff Fit')

# カットオフ以上の線形回帰
if not X_above.empty:
    model_above = np.polyfit(X_above, Y_above, 1)
    plt.plot(X_above, np.polyval(model_above, X_above), color='orange', label='Above Cutoff Fit')


plt.xlabel('X')
plt.ylabel('Y')
plt.title('RDD')
plt.legend()
plt.grid(True)
plt.show()


# パラメトリックモデルでの推定
# RDDの基本的なパラメトリックモデル: Y = alpha + tau*A + beta*X + gamma*A*X + epsilon
# カットオフからの距離 (X - cutoff) を共変量として使用する方が一般的だが、ここでは単純化

# 試験点数 X に定数項を追加
data["XA"] = data['A'] * data['X']
X_rdd = sm.add_constant(data[['A', 'X', 'XA']])
y_rdd = data['Y']

# OLSモデルをフィット
model_rdd = sm.OLS(y_rdd, X_rdd)
results_rdd = model_rdd.fit()

# 分析結果を表示
print("\nパラメトリックモデルによるRDD推定結果 (Y ~ A + X + XA):")
print(results_rdd.summary())

# 推定された処置効果 (Aの係数)
estimated_tau = results_rdd.params['A']
print(f"\n推定された平均処置効果 (tau): {estimated_tau:.2f}")

# 注: より洗練されたRDD分析では、カットオフからの距離、多項式回帰、バンド幅の選択などが重要になります。
# 上記は最も基本的な線形モデルでの推定例です。
